# Immune integration: scVI baseline comparison

Train standard `scvi.model.SCVI` on 7-dataset immune integration (bone marrow + PBMC).
Uses early stopping with stratified validation split.

**Model**: n_hidden=128, n_layers=2, n_latent=30, gene_likelihood=zinb, categorical_covariate_keys

**Input**: `results/immune_integration/adata_rna.h5ad` + `scrublet_results.csv`
**Output**: Trained model + latent representation + UMAP

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import scipy
import torch
import os

import scvi

scvi.settings.progress_bar_style = "tqdm"
torch.set_float32_matmul_precision("high")
rcParams["pdf.fonttype"] = 42

In [ ]:
# Papermill parameters
results_folder = (
    "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_integration_rna_scvi_baseline/"
)
marker_genes_csv = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/docs/notebooks/known_marker_genes.csv"
filtering_option = "compound"
skip_training = 0
wandb_project = "regularizedVI"
wandb_name = "immune_rna_scvi_baseline"
wandb_entity = None
wandb_notes = "scVI baseline: 128h/2L/30z/zinb, compound QC"
wandb_group = "immune_integration"
stratify_validation_key = "harmonized_annotation+batch"
early_stopping_min_delta_per_feature = 0.0002
early_stopping_patience = 10
max_epochs = 2000

## Data loading

In [ ]:
from data_loading_utils import load_atac_qc_metrics, load_scrublet_scores

data_folder = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_integration/"
rna_path = os.path.join(data_folder, "adata_rna.h5ad")
scrublet_path = os.path.join(data_folder, "scrublet_results.csv")
atac_qc_path = os.path.join(data_folder, "atac_qc_metrics.csv")

adata = sc.read_h5ad(rna_path)
print(f"Loaded: {adata.shape}")

# Load scrublet and ATAC QC using safe reindex utilities
adata = load_scrublet_scores(adata, scrublet_path)
adata = load_atac_qc_metrics(adata, atac_qc_path)

# Ensure counts in X
if "counts" in adata.layers:
    adata.X = adata.layers["counts"]

print(f"\nDatasets: {adata.obs['dataset'].nunique()}")
print(f"Batches: {adata.obs['batch'].nunique()}")
print(f"Donors: {adata.obs['donor'].nunique()}")

## QC summary

In [ ]:
from regularizedvi.utils import print_qc_summary

print_qc_summary(adata)

In [ ]:
from regularizedvi.utils import plot_qc_histograms

plot_qc_histograms(adata)

## Cell filtering

In [ ]:
if filtering_option == "adaptive":
    from data_loading_utils import adaptive_qc_filter

    adata = adaptive_qc_filter(
        adata,
        min_counts=900,
        min_genes=600,
        min_fragments=1500,
        per_dataset_quantile=0.20,
    )
elif filtering_option == "compound":
    from regularizedvi.utils import compound_qc_filter

    global_cutoffs = {
        "total_counts": (900, 80000),
        "n_genes": (600, 10000),
        "total_fragments": (1500, 100000),
        "mt_frac": (None, 0.15),
        "doublet_score": (None, 0.20),
    }
    per_study_cutoffs = {
        "bone_marrow": {
            "total_counts": (800, 80000),
            "n_genes": (600, 10000),
            "total_fragments": (2000, 100000),
            "mt_frac": (None, 0.10),
            "doublet_score": (None, 0.19),
        },
        "covid_pbmc": {
            "total_counts": (1500, 80000),
            "n_genes": (900, 10000),
            "total_fragments": (2000, 100000),
            "mt_frac": (None, 0.10),
            "doublet_score": (None, 0.18),
        },
        "crohns_pbmc": {
            "total_counts": (1500, 80000),
            "n_genes": (900, 10000),
            "total_fragments": (2000, 100000),
            "mt_frac": (None, 0.10),
            "doublet_score": (None, 0.18),
        },
        "infant_adult_spleen": {
            "total_counts": (1500, 80000),
            "n_genes": (900, 10000),
            "total_fragments": (2000, 100000),
            "mt_frac": (None, 0.10),
            "doublet_score": (None, 0.18),
        },
        "lung_spleen_gse319044": {
            "total_counts": (1200, 80000),
            "n_genes": (900, 10000),
            "total_fragments": (1500, 100000),
            "mt_frac": (None, 0.10),
            "doublet_score": (None, 0.18),
        },
        "neat_seq_cd4t": {
            "total_counts": (1200, 80000),
            "n_genes": (900, 10000),
            "total_fragments": (1500, 100000),
            "mt_frac": (None, 0.10),
            "doublet_score": (None, 0.25),
        },
        "pbmc_tea_seq": {
            "total_counts": (1200, 80000),
            "n_genes": (600, 10000),
            "total_fragments": (1500, 100000),
            "mt_frac": (None, 0.25),
            "doublet_score": (None, 0.18),
        },
    }
    adata = compound_qc_filter(
        adata,
        per_study_cutoffs=per_study_cutoffs,
        global_cutoffs=global_cutoffs,
    )
else:
    raise ValueError(f"Unknown filtering_option: {filtering_option!r}. Use 'adaptive' or 'compound'.")

## Gene filtering

In [ ]:
from regularizedvi.utils import filter_genes

selected = filter_genes(
    adata,
    cell_count_cutoff=15,
    cell_percentage_cutoff2=0.01,
    nonz_mean_cutoff=1.04,
)
adata = adata[:, selected].copy()
print(f"Selected {adata.n_vars:,} genes")

In [ ]:
adata.layers["counts"] = adata.X

In [ ]:
from regularizedvi.utils import coerce_papermill_params, finish_wandb, setup_wandb_logger

params = coerce_papermill_params(
    skip_training=(skip_training, bool),
    stratify_validation_key=(stratify_validation_key, "str_or_none"),
    early_stopping_min_delta_per_feature=(early_stopping_min_delta_per_feature, "float_or_none"),
    early_stopping_patience=(early_stopping_patience, int),
    max_epochs=(max_epochs, int),
    wandb_project=(wandb_project, "str_or_none"),
    wandb_name=(wandb_name, "str_or_none"),
    wandb_entity=(wandb_entity, "str_or_none"),
    wandb_notes=(wandb_notes, "str_or_none"),
    wandb_group=(wandb_group, "str_or_none"),
)
skip_training = params["skip_training"]
stratify_validation_key = params["stratify_validation_key"]
early_stopping_min_delta_per_feature = params["early_stopping_min_delta_per_feature"]
early_stopping_patience = params["early_stopping_patience"]
max_epochs = params["max_epochs"]
wandb_project = params["wandb_project"]
wandb_name = params["wandb_name"]
wandb_entity = params["wandb_entity"]
wandb_notes = params["wandb_notes"]
wandb_group = params["wandb_group"]

os.makedirs(results_folder, exist_ok=True)

wandb_loggers, wandb_run = setup_wandb_logger(
    wandb_project=wandb_project,
    wandb_name=wandb_name,
    wandb_entity=wandb_entity,
    wandb_notes=wandb_notes,
    wandb_group=wandb_group,
    config={
        "model": "scVI",
        "n_hidden": 128,
        "n_layers": 2,
        "n_latent": 30,
        "gene_likelihood": "zinb",
        "stratify_validation_key": stratify_validation_key,
        "early_stopping_min_delta_per_feature": early_stopping_min_delta_per_feature,
        "early_stopping_patience": early_stopping_patience,
        "filtering_option": filtering_option,
        "skip_training": skip_training,
    },
    results_folder=results_folder,
)

## scVI model setup and training

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    batch_key="batch",
    categorical_covariate_keys=["dataset", "donor"],
)

In [ ]:
model = scvi.model.SCVI(
    adata,
    n_hidden=128,
    n_layers=2,
    n_latent=30,
    gene_likelihood="zinb",
)
print(model)

In [ ]:
if not skip_training:
    import time
    from scvi.train import SaveCheckpoint

    checkpoint_cb = SaveCheckpoint(
        dirpath=f"{results_folder}/checkpoints",
        every_n_epochs=200,
        save_top_k=-1,
        filename="epoch-{epoch}",
    )

    # Note: early_stopping_min_delta_per_feature is not supported by scVI's trainer
    # so we use early_stopping_min_delta instead (absolute, not per-feature)
    _es_kwargs = {}
    if early_stopping_min_delta_per_feature is not None:
        _es_kwargs["early_stopping_min_delta"] = early_stopping_min_delta_per_feature * adata.n_vars

    _ds_kwargs = {"num_workers": 7}
    if stratify_validation_key is not None:
        from sklearn.model_selection import train_test_split

        keys = stratify_validation_key.split("+")
        strat_labels = adata.obs[keys[0]].astype(str)
        for k in keys[1:]:
            strat_labels = strat_labels + "_" + adata.obs[k].astype(str)
        counts = strat_labels.value_counts()
        rare = counts[counts < 2].index
        if len(rare) > 0:
            strat_labels = strat_labels.replace(rare, "_rare_")
            print(f"Merged {len(rare)} rare strata into '_rare_'")
        all_idx = np.arange(adata.n_obs)
        train_idx, val_idx = train_test_split(all_idx, test_size=0.1, stratify=strat_labels.values, random_state=42)
        _ds_kwargs["external_indexing"] = [train_idx, val_idx]
        print(f"Stratified split by {stratify_validation_key}: {len(train_idx)} train, {len(val_idx)} val")

    t0 = time.time()
    model.train(
        check_val_every_n_epoch=1,
        train_size=0.9,
        max_epochs=max_epochs,
        batch_size=1024,
        early_stopping=True,
        early_stopping_patience=early_stopping_patience,
        early_stopping_monitor="elbo_validation",
        enable_checkpointing=True,
        callbacks=[checkpoint_cb],
        logger=wandb_loggers,
        datasplitter_kwargs=_ds_kwargs,
        **_es_kwargs,
    )
    elapsed = time.time() - t0
    n_epochs = len(model.history_["elbo_train"])
    print(f"Training: {elapsed / 60:.1f} min, {n_epochs} epochs, {elapsed / n_epochs:.2f} s/epoch")
else:
    print("skip_training=True, skipping training")

## Results

In [ ]:
if not skip_training:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(model.history_["elbo_train"][80:])
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("ELBO")
    axes[0].set_title("ELBO (training)")
    axes[1].plot(model.history_["reconstruction_loss_train"][80:])
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Reconstruction loss")
    axes[1].set_title("Reconstruction loss (training)")
    plt.tight_layout()
    plt.show()

In [ ]:
ref_run_name = f"{results_folder}/model"
if not skip_training:
    model.save(ref_run_name, overwrite=True)

In [ ]:
if skip_training:
    import scvi

    model = scvi.model.SCVI.load(ref_run_name, adata=adata)
    print(f"Loaded model from {ref_run_name}")

latent = model.get_latent_representation()
adata.obsm["X_scVI"] = latent
print(f"Latent representation shape: {latent.shape}")

In [ ]:
k = 50
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=k, metric="euclidean")
sc.tl.umap(adata, min_dist=0.4, spread=1.3)
sc.tl.leiden(adata, resolution=12, flavor="igraph")

In [ ]:
# Technical covariates
rcParams["figure.figsize"] = 11, 11
sc.pl.umap(
    adata,
    color=["dataset", "batch", "site", "tissue", "age_group", "condition", "sex"],
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=1,
    vmin=0,
    vmax="p99.9",
    use_raw=False,
    legend_fontsize=8,
)

In [ ]:
# Cell annotation levels
rcParams["figure.figsize"] = 11, 11
sc.pl.umap(
    adata,
    color=["harmonized_annotation", "level_1", "level_2", "level_3"],
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=1,
    vmin=0,
    vmax="p99.9",
    use_raw=False,
    legend_fontsize=8,
)

In [ ]:
# Per-study integration highlight: each study in black, rest in light grey
rcParams["figure.figsize"] = 11, 11
datasets = adata.obs["dataset"].unique().tolist()
for ds in sorted(datasets):
    adata.obs["_highlight"] = pd.Categorical(
        np.where(adata.obs["dataset"] == ds, ds, "other"),
        categories=[ds, "other"],
    )
    sc.pl.umap(
        adata,
        color="_highlight",
        groups=[ds],
        na_color="lightgrey",
        palette={ds: "black", "other": "lightgrey"},
        size=1,
        title=f"Integration: {ds}",
    )
del adata.obs["_highlight"]

In [ ]:
# Per-study annotation plots: highlight one study's annotations, grey out the rest
rcParams["figure.figsize"] = 11, 11
annotation_levels = ["level_1", "level_2", "level_3", "harmonized_annotation"]
datasets = sorted(adata.obs["dataset"].unique().tolist())

for ds in datasets:
    mask = adata.obs["dataset"] == ds
    for level in annotation_levels:
        col_name = f"_{ds}_{level}"
        adata.obs[col_name] = pd.Categorical(np.where(mask, adata.obs[level].astype(str), np.nan))
        sc.pl.umap(
            adata,
            color=col_name,
            na_color="lightgrey",
            palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
            size=1,
            vmax="p99.9",
            use_raw=False,
            legend_fontsize=8,
            title=f"{ds} — {level}",
        )
        del adata.obs[col_name]

In [ ]:
# QC metrics on UMAP
rcParams["figure.figsize"] = 11, 11
qc_metrics = ["total_counts", "n_genes", "mt_frac", "doublet_score", "total_fragments"]
qc_present = [m for m in qc_metrics if m in adata.obs.columns]
sc.pl.umap(
    adata,
    color=qc_present,
    color_map="RdPu",
    ncols=3,
    size=1,
    vmax="p99.9",
)

In [ ]:
# Compute CPM-like normalized layer for marker gene plotting
import scipy.sparse

norm = adata.layers["counts"].astype(np.float32).copy()
if scipy.sparse.issparse(norm):
    row_sums = np.array(norm.sum(axis=1)).ravel()
    norm = norm.multiply(1.0 / row_sums[:, None]).tocsr()
else:
    row_sums = norm.sum(axis=1, keepdims=True)
    norm = norm / row_sums
norm = norm * 10000
adata.layers["norm_counts"] = norm
print(f"Added norm_counts layer: {norm.shape}")

In [ ]:
# Marker gene UMAPs — saved to results_folder/plots/ instead of embedding
import os
import matplotlib.pyplot as plt

plots_dir = os.path.join(results_folder, "plots", "marker_genes")
os.makedirs(plots_dir, exist_ok=True)

marker_df = pd.read_csv(marker_genes_csv)
categories = marker_df.groupby("category", sort=False)["gene"].apply(lambda x: list(dict.fromkeys(x)))

rcParams["figure.figsize"] = 11, 11
for category, gene_list in categories.items():
    present = [g for g in gene_list if g in adata.var["SYMBOL"].values]
    if not present:
        print(f"  {category}: no genes found in data, skipping")
        continue
    print(f"\n{category} ({len(present)}/{len(gene_list)} genes)")
    var_names = [adata.var_names[adata.var["SYMBOL"] == g][0] for g in present]
    sc.pl.umap(
        adata,
        color=var_names,
        layer="norm_counts",
        color_map="RdPu",
        ncols=5,
        size=1,
        vmax="p99.9",
        use_raw=False,
        title=present,
        show=False,
    )
    fig = plt.gcf()
    fig.savefig(os.path.join(plots_dir, f"{category}.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved to {plots_dir}/{category}.png")

In [ ]:
# Leiden clusters
rcParams["figure.figsize"] = 11, 11
sc.pl.umap(
    adata,
    color="leiden",
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=1,
    legend_fontsize=8,
)

In [ ]:
output_dir = f"{ref_run_name}/outputs/"
os.makedirs(output_dir, exist_ok=True)

X_scVI = pd.DataFrame(
    adata.obsm["X_scVI"],
    index=adata.obs_names,
    columns=range(adata.obsm["X_scVI"].shape[1]),
)
X_scVI.to_csv(f"{output_dir}/X_scVI.csv")

X_umap = pd.DataFrame(
    adata.obsm["X_umap"],
    index=adata.obs_names,
    columns=range(2),
)
X_umap.to_csv(f"{output_dir}/X_umap_k{k}.csv")

adata.obs[["leiden"]].to_csv(f"{output_dir}/leiden_k{k}.csv")

scipy.sparse.save_npz(f"{output_dir}/distances_euclidean_k{k}.npz", adata.obsp["distances"], compressed=True)
scipy.sparse.save_npz(f"{output_dir}/connectivities_euclidean_k{k}.npz", adata.obsp["connectivities"], compressed=True)

print(f"Outputs saved to {output_dir}")

In [ ]:
finish_wandb()